# forex_rl_v4 — Colab GPU training (multi-contributor)

Runs a slice of the full walk-forward sweep (12 folds x 3 seeds = 36
fold/seed combos, covering all 16 years of data on disk) as concurrent
`train.py` processes on ONE Colab GPU, writing to a Google Drive folder
SHARED across everyone contributing — so multiple people's sessions combine
into one pool of results instead of each person's work being stuck on
their own Drive.

**Nothing to download or upload** — the repo is public, so this notebook
clones the code and fetches the historical price data directly from
GitHub. All you need is a `MY_NAME` and the `SHARED_FOLDER` path (see the
project's Contribute tab, or ask the owner). Your results also push
themselves straight to GitHub in the background as you train — the owner's
dashboard picks you up automatically, nothing more to do after `Run all`.

**Before running:** `Runtime -> Change runtime type -> GPU`.

**If the session disconnects:** re-run the notebook from the top (`Runtime
-> Run all`). The claim cell hands you back the SAME shard indices (keyed
on `MY_NAME`), and each process's `--resume` skips fold/seed combos already
finished (by ANYONE, not just you) and resumes an interrupted fold from its
last mid-fold checkpoint. You'll lose whatever hadn't been auto-pushed yet
(up to ~5 minutes of progress), same as any other interruption.

In [ ]:
# === Config — set these before running ===
# MY_NAME: anything that identifies YOU uniquely (first name is fine) —
# used as the key in the shared claims registry so the system can tell your
# shards apart from everyone else's and hand you the SAME ones back if you
# re-run this cell (e.g. after a disconnect) instead of grabbing new ones.
MY_NAME = 'change-me'

# N_SHARDS_WANTED: how many shard slots to claim this session — one Colab
# GPU comfortably runs 4 in parallel (the project's standard allocation).
N_SHARDS_WANTED = 4

# TOTAL_SHARDS: total shard count across EVERYONE combined — the
# denominator for train.py's round-robin split. Must match what everyone
# else in the group is using. 36 = 12 folds x 3 seeds, i.e. one shard per
# fold/seed combo (the most parallelism this sweep can actually use).
TOTAL_SHARDS = 36

# SHARED_FOLDER: the Drive folder the project owner shared with you.
# Everyone points at the SAME folder so results combine into one pool
# instead of scattering across separate accounts.
SHARED_FOLDER = '/content/drive/MyDrive/forex_rl_v4_shared_results'

# ITERATIONS: PPO iterations per fold/seed combo.
ITERATIONS = 100

assert MY_NAME != 'change-me', 'set MY_NAME to something that identifies you first'
print(f'{MY_NAME}: wants {N_SHARDS_WANTED} of {TOTAL_SHARDS} total shards, '
      f'{ITERATIONS} iterations/combo, writing to {SHARED_FOLDER}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the code straight from GitHub — no zip, no manual upload. The repo
# is public, so this works with no authentication.
import os

PROJECT_DIR = '/content/martin'
REPO_URL = 'https://github.com/samdotbin/martin.git'

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
else:
    print(f'{PROJECT_DIR} already exists — pulling the latest instead of re-cloning.')
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}

In [ ]:
# data/raw's 182MB of historical CSVs isn't in git (see the repo's
# .gitignore) — it's attached to a GitHub Release instead. Downloads once;
# safe to re-run (skips if already present, e.g. after a disconnect where
# PROJECT_DIR survived but a fresh clone would still need this).
import urllib.request
import zipfile

DATA_URL = 'https://github.com/samdotbin/martin/releases/download/data-v1/forex_rl_v4_data_raw.zip'
data_dir = f'{PROJECT_DIR}/data/raw'

if os.path.isdir(data_dir) and len([f for f in os.listdir(data_dir) if f.endswith('.csv')]) >= 20:
    print(f'{data_dir} already has the data — skipping download.')
else:
    os.makedirs(data_dir, exist_ok=True)
    print('downloading historical price data (~43MB)...')
    tmp_zip = f'{data_dir}/_download.zip'
    urllib.request.urlretrieve(DATA_URL, tmp_zip)
    with zipfile.ZipFile(tmp_zip) as z:
        z.extractall(data_dir)
    os.remove(tmp_zip)
    n_csvs = len([f for f in os.listdir(data_dir) if f.endswith('.csv')])
    print(f'done — {n_csvs} CSV(s) in {data_dir}')

In [ ]:
# MetaTrader5 is Windows-only and gated by an environment marker in
# requirements.txt (`; platform_system == "Windows"`) — pip skips it
# automatically here, and nothing in the training path imports it.
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU visible. Runtime -> Change runtime type -> GPU, then re-run this cell."
)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Sanity check first — same coverage report you'd run locally.
!python data_pipeline.py

## Optional: bump ROLLOUT_N_ENVS for GPU

`config.ROLLOUT_N_ENVS` (default 32) sets how many environments are batched
into one model forward call. The model is small (2-5M params) and Colab GPUs
have plenty of headroom, so a higher value amortizes kernel-launch overhead
further. Edit `config.py` directly (or uncomment below) if you want to try
64 or 128 — there's no single right answer, and changing it does change the
exact (though not the statistical) outcome for a given seed, so pick a value
and keep it fixed for the whole sweep.

In [ ]:
# import config
# print('current ROLLOUT_N_ENVS:', config.ROLLOUT_N_ENVS)
# Edit config.py's ROLLOUT_N_ENVS value directly instead of monkey-patching
# here, since train.py runs as a subprocess below and won't see an in-notebook
# variable change.

In [ ]:
# Claim your shard indices from the shared registry — the system figures
# out which ones are still free, you don't need to be handed a range
# manually. Safe to re-run: idempotent for the SAME MY_NAME (returns your
# existing claims first, only grabs new ones for any shortfall).
import sys
sys.path.insert(0, PROJECT_DIR)
from scripts.claim_shards import claim

MY_SHARDS = claim(SHARED_FOLDER, TOTAL_SHARDS, N_SHARDS_WANTED, MY_NAME)
print(f'{MY_NAME}: claimed shards {MY_SHARDS}')

In [ ]:
import subprocess
import time

# Each of YOUR processes gets its own subfolder under the SHARED Drive
# folder (via env var overrides — see config.py) so concurrent processes —
# yours or anyone else's, in case sessions overlap — never read-modify-write
# the SAME RUN_MANIFEST.json at once. All processes share the one local
# code+data checkout (PROJECT_DIR) — data/raw is read-only.
os.makedirs(SHARED_FOLDER, exist_ok=True)

procs = []
for idx in MY_SHARDS:
    shard_dir = f'{SHARED_FOLDER}/shard{idx}'
    ckpt_dir = f'{shard_dir}/checkpoints'
    runs_dir = f'{shard_dir}/runs'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(runs_dir, exist_ok=True)

    env = os.environ.copy()
    env['FOREX_RL_CHECKPOINT_DIR'] = ckpt_dir
    env['FOREX_RL_RUNS_DIR'] = runs_dir

    log_path = f'{shard_dir}/train_log.txt'
    p = subprocess.Popen(
        ['python', 'train.py', '--resume', '--iterations', str(ITERATIONS),
         '--shard-index', str(idx), '--shard-count', str(TOTAL_SHARDS)],
        cwd=PROJECT_DIR, env=env,
        stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
    )
    procs.append({'idx': idx, 'proc': p, 'log': log_path})
    print(f'started shard {idx} (pid {p.pid}) -> {log_path}')
    time.sleep(2)  # stagger starts slightly so they don't all hit disk/data-loading at once

print(f'\n{len(procs)} shard(s) running in the background. Use the next cell '
      f'(re-run it any time) to check progress.')

In [ ]:
# Push YOUR results straight to GitHub automatically, in the background,
# every few minutes — this is what makes the owner's dashboard pick you up
# with no publish step on anyone's part. Uses a token the project owner
# placed in the shared folder (github_token.txt); nothing to set up here.
# Safe to re-run: starts one background thread, skips relaunching if
# already running.
import sys
import threading
import time

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
from scripts.push_to_github import push_all, read_shared_token

GITHUB_REPO = 'samdotbin/martin'
PUSH_INTERVAL_SECONDS = 300  # 5 minutes

def _auto_push_loop():
    token = read_shared_token(SHARED_FOLDER)
    while True:
        try:
            pushed = push_all(GITHUB_REPO, token, MY_NAME, SHARED_FOLDER, MY_SHARDS)
            if pushed:
                print(f'[{time.strftime("%H:%M:%S")}] pushed {len(pushed)} file(s) to GitHub')
        except Exception as e:
            print(f'[{time.strftime("%H:%M:%S")}] auto-push failed: {e}')
        time.sleep(PUSH_INTERVAL_SECONDS)

if '_auto_push_thread' in dir() and _auto_push_thread.is_alive():
    print('auto-push already running — not starting a second thread.')
else:
    _auto_push_thread = threading.Thread(target=_auto_push_loop, daemon=True)
    _auto_push_thread.start()
    print(f'background auto-push started — pushing to GitHub every {PUSH_INTERVAL_SECONDS // 60} min. '
          f'Leave this tab open; closing it stops the thread (your last push still counts).')

In [ ]:
# Re-run this cell any time to check progress. All "exited (code 0)" means
# every shard finished — move on to the merge cell at the bottom.
for entry in procs:
    rc = entry['proc'].poll()
    status = 'running' if rc is None else f'exited (code {rc})'
    print(f"shard {entry['idx']} (pid {entry['proc'].pid}): {status}")
    !tail -n 3 "{entry['log']}"
    print()

# Resource check — see the parallelism config cell for what to watch for.
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
!uptime

## Merge and archive

Every contributor's process wrote to their own `shard{i}/` subfolder under
the ONE shared Drive folder. This pulls in ALL `TOTAL_SHARDS` of them —
everyone's combined progress, not just yours — into one consolidated
`checkpoints/`+`runs/` in `PROJECT_DIR`, and zips that up for download.
Safe to run any time, by anyone with access to the shared folder, even
mid-training.

In [ ]:
import datetime

all_shard_dirs = ' '.join(f'{SHARED_FOLDER}/shard{idx}' for idx in range(TOTAL_SHARDS))
!python scripts/merge_shard_results.py {all_shard_dirs}

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
archive_path = f'/content/drive/MyDrive/forex_rl_v4_merged_results_{stamp}.zip'
!zip -rq "{archive_path}" checkpoints runs
print('wrote', archive_path)

## Live dashboard (optional)

Runs the project's dashboard (Contribute / Training / Performance / Price
chart / Agent replay / Compare seeds) right here on Colab, exposed via a
Cloudflare quick tunnel — a public URL you can open from any device, no
local PC needed. No account or token required; the URL only exists for
this session and stops working once the tunnel process ends.

Safe to run this at ANY point, including while the shards above are still
training — it merges whatever progress exists so far (additive, not
destructive) so the dashboard has something to show immediately. To refresh
with newer progress later, either re-run this cell, or use the dashboard's
own "Merge shard results" section (Training page) with the shard paths this
cell prints — no need to come back to the notebook at all once it's open.

In [ ]:
import re
import subprocess
import time

# 1. Pull in EVERY contributor's progress so far — safe mid-training,
# merge_shard_results.py is additive (dedupes by fold_id/seed/config, never
# deletes). Populates PROJECT_DIR/checkpoints+runs so the dashboard has
# something to show immediately.
shard_dirs = [f'{SHARED_FOLDER}/shard{idx}' for idx in range(TOTAL_SHARDS)]
print('Merging current progress from:')
for d in shard_dirs:
    print(' ', d)
!python scripts/merge_shard_results.py {' '.join(shard_dirs)}

# 2. Install cloudflared once per session (small binary, local disk is fine —
# it's a downloaded tool, not something that needs to survive a disconnect).
if not os.path.exists('/content/cloudflared'):
    print('\ninstalling cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

# 3. Launch Streamlit in the background — skip if a previous run of this
# cell already has it up, so re-running just re-merges + re-prints the URL
# instead of double-launching and fighting over port 8501.
if 'streamlit_proc' in dir() and streamlit_proc.poll() is None:
    print(f'\nStreamlit already running (pid {streamlit_proc.pid}) — not relaunching.')
else:
    streamlit_log = '/content/streamlit_log.txt'
    streamlit_proc = subprocess.Popen(
        ['streamlit', 'run', 'dashboard/streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'],
        cwd=PROJECT_DIR, stdout=open(streamlit_log, 'w'), stderr=subprocess.STDOUT,
    )
    print(f'\nstarted Streamlit (pid {streamlit_proc.pid}), waiting for it to boot...')
    time.sleep(8)

# 4. Launch the tunnel (same re-run guard) and pull its public URL from the log.
if 'tunnel_proc' in dir() and tunnel_proc.poll() is None:
    print(f'Tunnel already running (pid {tunnel_proc.pid}) — reusing existing URL below.')
else:
    tunnel_log = '/content/cloudflared_log.txt'
    tunnel_proc = subprocess.Popen(
        ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
        stdout=open(tunnel_log, 'w'), stderr=subprocess.STDOUT,
    )
    time.sleep(6)

with open(tunnel_log) as f:
    log_text = f.read()
match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
if match:
    print('\nDashboard URL:', match.group(0))
    print('(open from any device — stays live as long as this cell keeps running)')
else:
    print('\nTunnel URL not in the log yet — wait a few seconds and re-run just this cell.')
    print('If it keeps happening, check the raw log:', tunnel_log)

print('\nThis dashboard already shows EVERYONE\'s combined progress (it merged all')
print('TOTAL_SHARDS above). Re-run this cell any time to refresh with newer progress.')